# Lesson 09 Lab — PTQ Calibration Data: Sampling and Coverage

**Puzzle:** Can a small calibration set represent the activation ranges that production traffic will exercise?

This notebook keeps the RTX 5090 outputs from a complete run. Read the theory cells, make a prediction, and then use **Run All** on your own GPU.


## Why this matters

Post-training quantization freezes scales from examples. If those examples omit long prompts, a rare domain, or activation outliers, the quantizer can look excellent on calibration data and clip production traffic. Calibration quality is therefore a coverage problem before it is a sample-count problem.


## 0. Predict before running

1. Predict which calibration set minimizes clipping and which minimizes average rounding error on the mixed held-out set.
2. Explain why evaluation data must remain separate after scale selection.
3. List deployment strata that random sampling might under-represent.

For each answer, name the observation that would prove you wrong.


## 1. Name the concrete objects

A PTQ pipeline has a calibration distribution used to freeze quantization parameters and a disjoint evaluation distribution used to test the frozen result.

- Calibration estimates ranges or statistics; evaluation tests the frozen decision on held-out data.
- Rare domains and long sequences can dominate worst-case activation ranges.
- More samples do not help if sampling repeats the same narrow distribution.


## 2. Derive the mechanism

Max calibration protects observed extremes but can waste most codes; percentile or learned clipping trades a controlled tail for smaller steps. Either choice fails when the calibration set omits a deployment domain.

A max-range calibrator chooses `s=max(|x_cal|)/qmax`; a percentile calibrator deliberately clips a tail to shrink the step. Both estimate a property of the calibration distribution. Generalization fails when the deployment distribution has larger or differently located tails. More copies of the same narrow prompts reduce estimator noise but do not reduce distribution bias.

The held-out clipping fraction measures values outside the frozen representable range. RMSE measures the combined cost of clipped tails and quantization steps. Those objectives can disagree: an outlier-aware scale can avoid clipping yet waste resolution on most ordinary values.


## 3. Verify the execution environment

The next cell asserts CUDA availability, fixes the seed, locates the lesson, and prints a sanitized GPU/PyTorch/CUDA record. Check it before interpreting output.


In [1]:
from pathlib import Path
import json
import sys
import torch

chapter_rel = Path("chapters/01-mixed-precision-int4")
repo_root = next(
    p for p in [Path.cwd(), *Path.cwd().parents]
    if (p / chapter_rel / "support" / "lab_common.py").exists()
)
sys.path.insert(0, str(repo_root / chapter_rel / "support"))
from lab_common import (base_result, cuda_benchmark, environment_record,
                        error_metrics, require_cuda, save_result,
                        symmetric_quantize)

lesson_dir = repo_root / chapter_rel / "09-ptq-calibration"
device = require_cuda()
torch.manual_seed(2026 + 9)
environment = environment_record()
print(json.dumps(environment, indent=2))


{
  "gpu": "NVIDIA GeForce RTX 5090",
  "compute_capability": "12.0",
  "gpu_memory_gib": 31.358,
  "python": "3.12.13",
  "torch": "2.12.0",
  "cuda_runtime": "13.0"
}


## 4. Freeze the comparison

| Role | This run |
|---|---|
| Baseline | scales frozen from a narrow synthetic calibration distribution |
| Candidate | balanced and explicitly outlier-aware calibration sets |
| Held constant | INT8 formula, held-out mixed tensor, evaluation metrics, seed |
| Measurements | frozen scale, held-out clipping fraction, RMSE, MAE, cosine, max error |
| Evidence | `numerical-model` |

**Experiment:** Calibrate INT8 activation scales on narrow, balanced, and outlier-aware synthetic datasets, then evaluate all scales on a mixed held-out distribution.


## 5. Read the experiment code

The lab freezes scales from narrow, balanced, and outlier-aware sets and evaluates all three on one mixed held-out tensor.

The notebook creates three calibration populations, freezes one scale from each, and evaluates all of them on the same mixed held-out tensor. It never recomputes a scale on evaluation data. That makes the comparison a small distribution-shift test rather than a reconstruction demo.

The examples are synthetic so domain labels are controllable. A model study would replace them with stratified prompts and layer activation captures while preserving the same calibration/evaluation separation.

Only after these variables match the protocol should the cell be executed.


In [2]:
def sample(n, mode):
    x = torch.randn(n, 512, device=device)
    if mode == "shifted": x = x * 2.5 + 1.5
    if mode == "rare": x[:, ::64] *= 10
    return x
eval_x = torch.cat([sample(1024,"base"), sample(512,"shifted"), sample(128,"rare")])
cal_sets = {"narrow": sample(1024,"base"), "balanced": torch.cat([sample(512,"base"),sample(512,"shifted")]),
            "outlier_aware": torch.cat([sample(448,"base"),sample(448,"shifted"),sample(128,"rare")])}
rows = {}
for name, cal in cal_sets.items():
    scale = cal.abs().max() / 127; q = torch.round(eval_x/scale).clamp(-128,127); dq=q*scale
    rows[name] = {"scale": round(scale.item(),8), "clipping_fraction": round((eval_x.abs()>127*scale).float().mean().item(),8),
                  "error": error_metrics(eval_x,dq)}
result=base_result(9,"numerical-model"); result.update({"evaluation_shape":list(eval_x.shape),"calibration_results":rows,
    "conclusion":"Held-out coverage, not calibration reconstruction, determined clipping and error."})


## 6. Read the retained RTX 5090 result

**Recorded environment:** NVIDIA GeForce RTX 5090; compute capability 12.0; PyTorch 2.12.0; CUDA runtime 13.0.

| Measured field | Checked-in value |
|---|---:|
| Narrow clipping | 2.6478% |
| Narrow RMSE | 0.317395 |
| Balanced clipping | 0.0250% |
| Balanced RMSE | 0.085046 |
| Outlier-aware clipping | 0.0000% |
| Outlier-aware RMSE | 0.077629 |


## 7. Interpret rather than merely print

The narrow scale clipped 2.647752% of held-out values and produced RMSE 0.317395 with a max error of 27.9039. Balanced calibration reduced clipping to 0.025001% and RMSE to 0.085046. Outlier-aware calibration eliminated clipping, but its larger scale raised MAE to 0.067231; its RMSE, 0.077629, remained slightly better because it avoided catastrophic tail errors.

There is no universally best row without a deployment objective. If tail failures are unacceptable, the outlier-aware scale wins this probe. If average small-value resolution dominates, a clipped or mixed policy may be preferable.

**Inspection rule:** Compare held-out clipping rate and error, not calibration-set reconstruction error.


## 8. Keep the evidence label honest

This run is labeled **`numerical-model`**. The CUDA numerical experiment isolates an algorithmic mechanism. It is not the paper's complete implementation and does not establish a production kernel speedup.

The next cell writes the complete structured result; its existing saved output is part of the checked-in evidence.


In [3]:
artifact_path = save_result(result, lesson_dir)
print(json.dumps(result, indent=2, sort_keys=True))
print("Saved: artifacts/rtx5090-result.json")


{
  "calibration_results": {
    "balanced": {
      "clipping_fraction": 0.00025001,
      "error": {
        "cosine": 0.99894321,
        "mae": 0.0260779,
        "max_abs": 20.09401703,
        "rmse": 0.08504558
      },
      "scale": 0.10047337
    },
    "narrow": {
      "clipping_fraction": 0.02647752,
      "error": {
        "cosine": 0.98645717,
        "mae": 0.04251424,
        "max_abs": 27.9039135,
        "rmse": 0.31739485
      },
      "scale": 0.03945856
    },
    "outlier_aware": {
      "clipping_fraction": 0.0,
      "error": {
        "cosine": 0.99911618,
        "mae": 0.06723144,
        "max_abs": 0.13448334,
        "rmse": 0.07762861
      },
      "scale": 0.26896697
    }
  },
  "conclusion": "Held-out coverage, not calibration reconstruction, determined clipping and error.",
  "environment": {
    "compute_capability": "12.0",
    "cuda_runtime": "13.0",
    "gpu": "NVIDIA GeForce RTX 5090",
    "gpu_memory_gib": 31.358,
    "python": "3.12.13",
   

## 9. Make the bounded decision

> Choose calibration data by coverage of deployment modes, and keep it separate from the regression set.

**Acceptance/rollback:** Publish sampling rules, lengths/domains, seed, statistic, sample count, and held-out clipping/error. Never tune the range on the same examples used for the final quality gate.

**Failure analysis:** Tuning percentiles on the final regression set leaks evaluation into calibration. Reporting only mean error can hide rare catastrophic clipping, while reporting only max error can let one outlier consume the entire code range. Coverage metadata—domain, length, language, tool use, and frequency—is part of the quantization artifact.


## 10. Extend the evidence

Build a stratified calibration manifest for real prompts and compare random, balanced, and tail-oversampled selections at fixed sample count. Evaluate per-layer clipping and task slices on a disjoint set, then test whether the selected scale policy remains stable across model revisions.

The full derivation, reproduction command, evidence boundary and primary references are in [`README.md`](README.md).
